# Unit Test: Circular Supercell Model

This notebook implements a redesigned **circular supercell** model and runs 3 sanity-check scenarios with 2 supercells each.

### Overview
- Each supercell is defined by a center, gradient-zone radius, max leg half-length, and scientific weight.
- The aircraft executes a **cross-pattern** scan: two perpendicular legs through the center.
- Optional **obstacles** (convex polygons) may block certain cross orientations.
- Optional **sensitivity points** are mandatory waypoints near the boundary that must be visited after the cross scan.
- A brute-force optimizer selects which cells to visit and with what orientation (theta, L) to maximize total scientific score within a distance budget.

## Cell 1: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from itertools import permutations, combinations, product as iproduct
from math import cos, sin, radians, sqrt

np.set_printoptions(precision=3, suppress=True)
print('Imports OK')

## Cell 2: Geometry Utilities

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GEOMETRY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def uvec(angle: float) -> np.ndarray:
    """Unit vector at given angle (radians)."""
    return np.array([cos(angle), sin(angle)])


def cross2d(v: np.ndarray, w: np.ndarray) -> float:
    """2D cross product of vectors v and w."""
    return float(v[0] * w[1] - v[1] * w[0])


def ensure_ccw(verts: List[np.ndarray]) -> List[np.ndarray]:
    """Return vertices in counter-clockwise order."""
    pts = [np.asarray(v, dtype=float) for v in verts]
    n = len(pts)
    area2 = 0.0
    for i in range(n):
        j = (i + 1) % n
        area2 += pts[i][0] * pts[j][1]
        area2 -= pts[j][0] * pts[i][1]
    if area2 < 0:
        pts = pts[::-1]  # reverse to CCW
    return pts


def segments_intersect_proper(p1: np.ndarray, p2: np.ndarray,
                               p3: np.ndarray, p4: np.ndarray) -> bool:
    """
    True if segment p1-p2 properly intersects segment p3-p4
    (excludes shared endpoints and collinear overlaps).
    """
    EPS = 1e-9
    d1 = p2 - p1
    d2 = p4 - p3
    denom = cross2d(d1, d2)
    if abs(denom) < EPS:
        return False  # parallel or collinear
    diff = p3 - p1
    t = cross2d(diff, d2) / denom
    u = cross2d(diff, d1) / denom
    return (EPS < t < 1.0 - EPS) and (EPS < u < 1.0 - EPS)


def point_in_convex_polygon(pt: np.ndarray, verts: List[np.ndarray]) -> bool:
    """
    True if pt is strictly inside the CCW convex polygon defined by verts.
    Uses the cross-product sign test for each edge.
    """
    EPS = 1e-9
    n = len(verts)
    for i in range(n):
        edge = verts[(i + 1) % n] - verts[i]
        to_pt = pt - verts[i]
        if cross2d(edge, to_pt) <= EPS:
            return False
    return True


def segment_blocked_by_polygon(p1: np.ndarray, p2: np.ndarray,
                                verts: List[np.ndarray]) -> bool:
    """
    True if segment p1-p2 passes through the interior of the convex polygon.
    Checks proper edge crossings and whether the segment midpoint is inside.
    Tolerance: 1e-9 for numerical stability.
    """
    n = len(verts)
    for i in range(n):
        j = (i + 1) % n
        if segments_intersect_proper(p1, p2, verts[i], verts[j]):
            return True
    mid = 0.5 * (p1 + p2)
    if point_in_convex_polygon(mid, verts):
        return True
    return False


def path_length(waypoints: List[np.ndarray]) -> float:
    """Total Euclidean length of a sequence of waypoints."""
    total = 0.0
    for i in range(len(waypoints) - 1):
        total += np.linalg.norm(waypoints[i + 1] - waypoints[i])
    return total


def obstacle_free_path(start: np.ndarray, end: np.ndarray,
                        polygon_verts: List[np.ndarray]) -> List[np.ndarray]:
    """
    Returns shortest obstacle-free path from start to end around a single
    convex polygon.

    Strategy:
    - If direct path is clear, return [start, end].
    - Otherwise, try all subsets of polygon vertices as intermediate waypoints
      via CW and CCW arc chains; pick the shortest valid route.
    - Fallback: return direct path if no clear routing found.
    """
    start = np.asarray(start, dtype=float)
    end   = np.asarray(end,   dtype=float)
    verts = [np.asarray(v, dtype=float) for v in polygon_verts]

    def is_clear(a, b):
        return not segment_blocked_by_polygon(a, b, verts)

    if is_clear(start, end):
        return [start, end]

    n = len(verts)
    best_path = None
    best_dist = float('inf')

    # Try routing via every single vertex
    for v in verts:
        if is_clear(start, v) and is_clear(v, end):
            d = np.linalg.norm(v - start) + np.linalg.norm(end - v)
            if d < best_dist:
                best_dist = d
                best_path = [start, v, end]

    # Try routing via consecutive vertex arcs (CW and CCW)
    for start_idx in range(n):
        for length in range(1, n):
            # CCW arc
            arc_ccw  = [verts[(start_idx + k) % n] for k in range(length + 1)]
            candidate = [start] + arc_ccw + [end]
            if all(is_clear(candidate[k], candidate[k + 1])
                   for k in range(len(candidate) - 1)):
                d = path_length(candidate)
                if d < best_dist:
                    best_dist = d
                    best_path = candidate
            # CW arc
            arc_cw   = [verts[(start_idx - k) % n] for k in range(length + 1)]
            candidate = [start] + arc_cw + [end]
            if all(is_clear(candidate[k], candidate[k + 1])
                   for k in range(len(candidate) - 1)):
                d = path_length(candidate)
                if d < best_dist:
                    best_dist = d
                    best_path = candidate

    return best_path if best_path is not None else [start, end]


def multi_obstacle_free_path(start: np.ndarray, end: np.ndarray,
                              obstacles: List[List[np.ndarray]]) -> List[np.ndarray]:
    """
    Route from start to end avoiding multiple convex polygon obstacles.
    Applies obstacle_free_path iteratively for each obstacle.
    Collects ALL obstacles from ALL cells in the scenario.
    Returns list of waypoints.
    """
    if not obstacles:
        return [np.asarray(start, dtype=float), np.asarray(end, dtype=float)]

    current_path = [np.asarray(start, dtype=float), np.asarray(end, dtype=float)]

    for obs_verts in obstacles:
        obs_verts = [np.asarray(v, dtype=float) for v in obs_verts]
        new_path  = [current_path[0]]
        for i in range(len(current_path) - 1):
            sub_path = obstacle_free_path(current_path[i], current_path[i + 1], obs_verts)
            new_path.extend(sub_path[1:])  # skip first (already in new_path)
        current_path = new_path

    return current_path


print('Geometry utilities loaded.')

## Cell 3: CircularSupercell Dataclass

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CIRCULAR SUPERCELL DATACLASS
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class CircularSupercell:
    """
    Defines a circular supercell target.

    Fixed geometry (stored in dataclass):
      center             : 2D position of cell center
      radius             : gradient-zone radius; cross legs must extend beyond this
      L_max              : maximum leg half-length (hard upper bound)
      weight             : scientific priority / score value
      name               : human-readable label
      sensitivity_points : mandatory waypoints visited after the cross scan (may be empty)
      obstacle           : CCW convex polygon vertices that block certain cross
                           orientations, or None

    Optimization variables (NOT stored here):
      theta : scan direction in THETA_GRID = {0deg, 15deg, ..., 165deg}
      L     : leg half-length, must satisfy radius < L <= L_max

    Cross-pattern for a given (theta, L):
      P0 = center - L * uvec(theta)          # leg 1 endpoint A
      P1 = center + L * uvec(theta)          # leg 1 endpoint B
      P2 = center - L * uvec(theta + pi/2)   # leg 2 endpoint A
      P3 = center + L * uvec(theta + pi/2)   # leg 2 endpoint B

      Flight path:  P0 -> center -> P1 -> P2 -> center -> P3
                 or reversed:  P3 -> center -> P2 -> P1 -> center -> P0

      Entry = P0 if |from_point - P0| <= |from_point - P3|, else P3
      Internal flight distance = (4 + sqrt(2)) * L

    Score: cell earns its weight ONLY IF cross is executed AND all
    sensitivity_points are visited.
    """
    center:             np.ndarray
    radius:             float
    L_max:              float
    weight:             float
    name:               str
    sensitivity_points: List[np.ndarray] = field(default_factory=list)
    obstacle:           Optional[List[np.ndarray]] = None

    def __post_init__(self):
        self.center = np.asarray(self.center, dtype=float)
        self.sensitivity_points = [
            np.asarray(sp, dtype=float) for sp in self.sensitivity_points
        ]
        if self.obstacle is not None:
            # Guarantee CCW orientation for point_in_convex_polygon
            self.obstacle = ensure_ccw(
                [np.asarray(v, dtype=float) for v in self.obstacle]
            )

    def cross_endpoints(self, theta: float, L: float):
        """Returns (P0, P1, P2, P3) for the cross pattern."""
        u1 = uvec(theta)
        u2 = uvec(theta + np.pi / 2)
        P0 = self.center - L * u1
        P1 = self.center + L * u1
        P2 = self.center - L * u2
        P3 = self.center + L * u2
        return P0, P1, P2, P3

    def cross_waypoints(
            self, theta: float, L: float, from_point: np.ndarray
    ) -> Tuple[np.ndarray, List[np.ndarray], np.ndarray]:
        """
        Returns (entry, waypoints, exit) choosing entry to minimize
        transit distance from from_point.
        """
        P0, P1, P2, P3 = self.cross_endpoints(theta, L)
        c = self.center
        if np.linalg.norm(from_point - P0) <= np.linalg.norm(from_point - P3):
            return P0, [P0, c, P1, P2, c, P3], P3
        else:
            return P3, [P3, c, P2, P1, c, P0], P0

    def cross_blocked(self, theta: float, L: float) -> bool:
        """
        Returns True if ANY of the 4 half-legs of the cross pattern is
        blocked by the cell's own obstacle polygon.
        """
        if self.obstacle is None:
            return False
        P0, P1, P2, P3 = self.cross_endpoints(theta, L)
        c = self.center
        return any(
            segment_blocked_by_polygon(c, p, self.obstacle)
            for p in [P0, P1, P2, P3]
        )

    def internal_distance(self, L: float) -> float:
        """Internal cross-pattern flight distance = (4 + sqrt(2)) * L."""
        return (4.0 + sqrt(2.0)) * L


print('CircularSupercell dataclass loaded.')

## Cell 4: Route and Optimizer Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ROUTE AND OPTIMIZER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def collect_obstacles(cells: List[CircularSupercell]) -> List[List[np.ndarray]]:
    """Collect all obstacle polygon vertex lists from ALL cells in the scenario."""
    return [c.obstacle for c in cells if c.obstacle is not None]


def route_through_sensitivity_points(
        start: np.ndarray,
        sens_pts: List[np.ndarray],
        all_obstacles: List[List[np.ndarray]]
) -> Tuple[float, np.ndarray, List[np.ndarray]]:
    """
    Find shortest obstacle-avoiding route through all sensitivity points
    starting at `start`.  Tries all permutations (<= 5! = 120).

    Returns:
        (best_total_dist, final_position, waypoints_list)
    waypoints_list does NOT include start.
    """
    start = np.asarray(start, dtype=float)
    if not sens_pts:
        return 0.0, start, []

    best_dist = float('inf')
    best_wpts = None
    best_end  = None

    for perm in permutations(range(len(sens_pts))):
        ordered = [sens_pts[i] for i in perm]
        wpts  = []
        pos   = start
        total = 0.0
        for sp in ordered:
            seg    = multi_obstacle_free_path(pos, sp, all_obstacles)
            total += path_length(seg)
            wpts.extend(seg[1:])  # skip first point (already at pos)
            pos = seg[-1]
        if total < best_dist:
            best_dist = total
            best_wpts = wpts
            best_end  = pos

    return best_dist, best_end, best_wpts


def compute_full_route(
        visit_order: List[int],
        cells: List[CircularSupercell],
        thetas: List[float],
        Ls: List[float],
        BASE: np.ndarray
) -> Tuple[float, List[np.ndarray]]:
    """
    Compute total distance and full waypoint list for a given visit order.

    Full route:
      BASE -> [transit] -> Cell_i_entry -> cross_i -> Cell_i_exit
           -> [sensitivity_i routing] -> ... -> BASE

    Returns:
        (total_distance, full_waypoints)
    """
    BASE     = np.asarray(BASE, dtype=float)
    all_obs  = collect_obstacles(cells)
    total_dist = 0.0
    full_wpts  = [BASE.copy()]
    pos        = BASE.copy()

    for idx in visit_order:
        cell  = cells[idx]
        theta = thetas[idx]
        L     = Ls[idx]

        # Choose entry point closer to current position
        P0, P1, P2, P3 = cell.cross_endpoints(theta, L)
        entry = P0 if np.linalg.norm(pos - P0) <= np.linalg.norm(pos - P3) else P3

        # Transit to entry (obstacle-avoiding)
        transit = multi_obstacle_free_path(pos, entry, all_obs)
        total_dist += path_length(transit)
        full_wpts.extend(transit[1:])
        pos = transit[-1]

        # Execute cross pattern
        _, cross_wpts, exit_pt = cell.cross_waypoints(theta, L, pos)
        for i in range(len(cross_wpts) - 1):
            total_dist += np.linalg.norm(cross_wpts[i + 1] - cross_wpts[i])
        full_wpts.extend(cross_wpts[1:])
        pos = exit_pt

        # Route through sensitivity points (if any)
        if cell.sensitivity_points:
            s_dist, s_end, s_wpts = route_through_sensitivity_points(
                pos, cell.sensitivity_points, all_obs
            )
            total_dist += s_dist
            full_wpts.extend(s_wpts)
            pos = s_end

    # Return to BASE
    return_path = multi_obstacle_free_path(pos, BASE, all_obs)
    total_dist += path_length(return_path)
    full_wpts.extend(return_path[1:])

    return total_dist, full_wpts


def optimize(
        cells: List[CircularSupercell],
        BASE: np.ndarray,
        BUDGET: float,
        THETA_GRID: np.ndarray,
        N_L_LEVELS: int = 6
) -> dict:
    """
    Brute-force optimizer: find best subset/order of cells to visit.

    For each non-empty subset of cells (every permutation order),
    for each combination of theta values and L levels:
      - Skip configs where cross intersects own obstacle.
      - Compute route distance; skip if > BUDGET.
      - Maximize sum of weights of visited cells.
      - Tie-break: minimize distance.

    Prints progress every 500 iterations when total > 2000.

    Returns:
        dict with keys: score, dist, order, thetas, Ls, waypoints
    """
    BASE    = np.asarray(BASE, dtype=float)
    n_cells = len(cells)

    L_grids = [
        np.linspace(c.radius + 10, c.L_max, N_L_LEVELS)
        for c in cells
    ]

    best = {
        'score':     -1.0,
        'dist':      float('inf'),
        'order':     [],
        'thetas':    [0.0] * n_cells,
        'Ls':        [c.radius + 10 for c in cells],
        'waypoints': []
    }

    # All non-empty subsets in all permutation orders
    all_subsets = []
    for r in range(1, n_cells + 1):
        for subset in combinations(range(n_cells), r):
            for perm in permutations(subset):
                all_subsets.append(list(perm))

    iter_count = 0

    for order in all_subsets:
        theta_options = [THETA_GRID        for _   in order]
        L_options     = [L_grids[idx]      for idx in order]

        for theta_combo in iproduct(*theta_options):
            for L_combo in iproduct(*L_options):
                iter_count += 1

                # Build per-cell theta/L arrays (index = cell index)
                thetas_all = [0.0] * n_cells
                Ls_all     = [cells[i].radius + 10 for i in range(n_cells)]
                for k, idx in enumerate(order):
                    thetas_all[idx] = theta_combo[k]
                    Ls_all[idx]     = L_combo[k]

                # Feasibility: check obstacle blocking for each cell in order
                if any(cells[order[k]].cross_blocked(theta_combo[k], L_combo[k])
                       for k in range(len(order))):
                    continue

                dist, wpts = compute_full_route(
                    order, cells, thetas_all, Ls_all, BASE
                )
                if dist > BUDGET:
                    continue

                score = sum(cells[idx].weight for idx in order)

                if score > best['score'] or (
                        score == best['score'] and dist < best['dist']):
                    best = {
                        'score':     score,
                        'dist':      dist,
                        'order':     order,
                        'thetas':    thetas_all,
                        'Ls':        Ls_all,
                        'waypoints': wpts
                    }

    print(f'  Optimizer finished: {iter_count} iterations evaluated.')
    return best


print('Route and optimizer functions loaded.')

## Cell 5: Visualization

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

CELL_COLORS = ['steelblue', 'darkorange', 'forestgreen', 'mediumpurple']


def draw_scenario(
        ax,
        cells: List[CircularSupercell],
        BASE: np.ndarray,
        solution: Optional[dict] = None,
        title: str = ''
):
    """
    Draw the full scenario on the given Matplotlib axes.

    Elements drawn:
      - Each cell: filled circle (alpha=0.12) + boundary circle
      - Obstacle (if any): filled red polygon (alpha=0.3) with 'OBS' label
      - Sensitivity points: orange stars with labels S1, S2, ...
      - BASE: black square marker
      - If solution provided:
          - Full route: green line with directional arrow
          - Cross patterns: blue lines with square endpoint markers
          - Visit order number and theta annotation per cell
      - Title includes score and distance if solution provided.
    """
    BASE = np.asarray(BASE, dtype=float)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3, linestyle='--')

    # ── Draw cells ──────────────────────────────────────────────────────────
    for i, cell in enumerate(cells):
        color = CELL_COLORS[i % len(CELL_COLORS)]
        cx, cy = cell.center

        ax.add_patch(plt.Circle((cx, cy), cell.radius,
                                color=color, alpha=0.12, zorder=1))
        ax.add_patch(plt.Circle((cx, cy), cell.radius,
                                fill=False, edgecolor=color,
                                linewidth=1.5, zorder=2))
        ax.text(cx, cy + cell.radius + 8, cell.name,
                ha='center', va='bottom', fontsize=9,
                color=color, fontweight='bold')
        ax.plot(cx, cy, 'o', color=color, markersize=4, zorder=3)

        # Obstacle
        if cell.obstacle is not None:
            poly_xy = np.array(cell.obstacle)
            ax.add_patch(mpatches.Polygon(
                poly_xy, closed=True,
                facecolor='red', edgecolor='darkred',
                alpha=0.30, linewidth=1.5, zorder=4
            ))
            ax.text(poly_xy[:, 0].mean(), poly_xy[:, 1].mean(),
                    'OBS', ha='center', va='center',
                    fontsize=7, color='darkred', fontweight='bold', zorder=5)

        # Sensitivity points
        for j, sp in enumerate(cell.sensitivity_points):
            ax.plot(sp[0], sp[1], '*', color='darkorange',
                    markersize=12, zorder=6,
                    markeredgecolor='black', markeredgewidth=0.5)
            ax.text(sp[0] + 5, sp[1] + 5, f'S{j + 1}',
                    fontsize=8, color='darkorange',
                    fontweight='bold', zorder=7)

    # ── Draw cross patterns ──────────────────────────────────────────────────
    if solution is not None and solution.get('order'):
        for rank, idx in enumerate(solution['order']):
            cell  = cells[idx]
            theta = solution['thetas'][idx]
            L     = solution['Ls'][idx]
            P0, P1, P2, P3 = cell.cross_endpoints(theta, L)
            c = cell.center

            ax.plot([P0[0], P1[0]], [P0[1], P1[1]],
                    '-', color='royalblue', linewidth=2.0, zorder=8)
            ax.plot([P2[0], P3[0]], [P2[1], P3[1]],
                    '-', color='royalblue', linewidth=2.0, zorder=8)
            for pt in [P0, P1, P2, P3]:
                ax.plot(pt[0], pt[1], 's', color='royalblue',
                        markersize=5, zorder=9)

            ax.text(c[0] - 5, c[1] - 14, f'#{rank + 1}',
                    fontsize=10, color='royalblue',
                    fontweight='bold', zorder=10)
            ax.text(c[0] + 5, c[1] - 22,
                    f'theta={round(np.degrees(theta), 1)}deg',
                    fontsize=7, color='royalblue', zorder=10)

    # ── Draw full route ──────────────────────────────────────────────────────
    if solution is not None and solution.get('waypoints'):
        wpts = solution['waypoints']
        xs = [w[0] for w in wpts]
        ys = [w[1] for w in wpts]
        ax.plot(xs, ys, '-', color='limegreen', linewidth=1.5,
                alpha=0.80, zorder=7, label='Route')
        # Directional arrow near the midpoint of the route
        if len(wpts) >= 2:
            mid = len(wpts) // 2
            dx  = wpts[mid][0] - wpts[mid - 1][0]
            dy  = wpts[mid][1] - wpts[mid - 1][1]
            d   = sqrt(dx ** 2 + dy ** 2)
            if d > 1e-6:
                ax.annotate(
                    '',
                    xy=(wpts[mid][0], wpts[mid][1]),
                    xytext=(wpts[mid][0] - dx / d * 20,
                            wpts[mid][1] - dy / d * 20),
                    arrowprops=dict(arrowstyle='->', color='green', lw=2.0),
                    zorder=11
                )

    # ── BASE marker ─────────────────────────────────────────────────────────
    ax.plot(BASE[0], BASE[1], 's', color='black',
            markersize=10, zorder=12, label='BASE')
    ax.text(BASE[0] + 8, BASE[1] + 8, 'BASE',
            fontsize=9, color='black', fontweight='bold', zorder=13)

    # ── Title ───────────────────────────────────────────────────────────────
    if solution is not None and solution.get('order'):
        full_title = (
            f"{title}\n"
            f"Score={solution['score']:.2f}  "
            f"Dist={solution['dist']:.1f} km  "
            f"Visited: {solution['order']}"
        )
    else:
        full_title = title
    ax.set_title(full_title, fontsize=11, pad=10)
    ax.set_xlabel('x (km)')
    ax.set_ylabel('y (km)')

    # ── Auto-scale with margin ───────────────────────────────────────────────
    all_pts = [BASE]
    for cell in cells:
        all_pts.append(cell.center)
        if cell.obstacle:
            all_pts.extend(cell.obstacle)
        all_pts.extend(cell.sensitivity_points)
    if solution and solution.get('waypoints'):
        all_pts.extend(solution['waypoints'])
    all_pts = np.array(all_pts)
    margin  = 60
    ax.set_xlim(all_pts[:, 0].min() - margin, all_pts[:, 0].max() + margin)
    ax.set_ylim(all_pts[:, 1].min() - margin, all_pts[:, 1].max() + margin)
    ax.legend(loc='lower right', fontsize=8)


print('Visualization functions loaded.')

---
## Scenario 1: No Obstacles, No Sensitivity Points

Both cells are fully free: no obstacles restrict any cross orientation, and no mandatory waypoints are required.  
The optimizer chooses theta and L for each cell purely to minimize total travel distance while visiting both cells.

**Expected**: both cells visited (score = 3.5), theta chosen to minimize route length.

In [ ]:
# ─── Scenario 1 Setup ────────────────────────────────────────────────────────

BASE       = np.array([50.0, 300.0])
THETA_GRID = np.radians(np.arange(0, 180, 15))  # 12 values: 0, 15, ..., 165 deg

cells_s1 = [
    CircularSupercell(
        center=np.array([200.0, 300.0]),
        radius=50.0,
        L_max=130.0,
        weight=2.0,
        name='TPV-1',
        sensitivity_points=[],
        obstacle=None
    ),
    CircularSupercell(
        center=np.array([500.0, 280.0]),
        radius=45.0,
        L_max=120.0,
        weight=1.5,
        name='TPV-2',
        sensitivity_points=[],
        obstacle=None
    ),
]

# Budget set above the minimum feasible distance to visit both cells (~1368 km)
BUDGET_S1 = 1500.0

print('Scenario 1 cells defined.')
for c in cells_s1:
    print(f'  {c.name}: center={c.center}, radius={c.radius}, '
          f'L_max={c.L_max}, weight={c.weight}')

In [ ]:
# ─── Scenario 1 Optimize ─────────────────────────────────────────────────────

print('Running optimizer for Scenario 1...')
sol_s1 = optimize(cells_s1, BASE, BUDGET_S1, THETA_GRID, N_L_LEVELS=6)
print(f'  Best score  : {sol_s1["score"]:.2f}')
print(f'  Best dist   : {sol_s1["dist"]:.2f} km')
print(f'  Visit order : {sol_s1["order"]}')
print(f'  Thetas (deg): {[round(np.degrees(t), 1) for t in sol_s1["thetas"]]}')
print(f'  Ls          : {[round(L, 2) for L in sol_s1["Ls"]]}')

In [ ]:
# ─── Scenario 1 Plot ─────────────────────────────────────────────────────────

fig1, ax1 = plt.subplots(figsize=(9, 6))
draw_scenario(
    ax1, cells_s1, BASE, solution=sol_s1,
    title='Scenario 1: No Obstacles, No Sensitivity Points'
)
plt.tight_layout()
plt.savefig(
    '/home/yuanpei_cao/aerial-met-survey-optimizer/figures/unit_test_s1.png',
    dpi=120, bbox_inches='tight'
)
plt.show()
print('Figure saved.')

In [ ]:
# ─── Scenario 1 Summary Table ─────────────────────────────────────────────────

print('=' * 55)
print('SCENARIO 1 SUMMARY')
print('=' * 55)
print(f'{"Cell":<8} {"Visited":<9} {"Theta (deg)":<13} {"L (km)":<10} {"Weight":<8}')
print('-' * 55)
for i, cell in enumerate(cells_s1):
    visited   = 'YES' if i in sol_s1['order'] else 'NO'
    theta_deg = round(np.degrees(sol_s1['thetas'][i]), 1)
    L_val     = round(sol_s1['Ls'][i], 2)
    print(f'{cell.name:<8} {visited:<9} {theta_deg:<13} {L_val:<10} {cell.weight:<8}')
print('-' * 55)
print(f'{"TOTAL":<8} {"":<9} {"":<13} {"":<10} {sol_s1["score"]:<8.2f}')
print(f'Total route distance : {sol_s1["dist"]:.2f} km')
print(f'Budget               : {BUDGET_S1:.0f} km')
print(f'Budget utilization   : {sol_s1["dist"] / BUDGET_S1 * 100:.1f}%')
print('=' * 55)

---
## Scenario 2: With Obstacles, No Sensitivity Points

Each cell has a convex polygon obstacle that blocks a range of cross orientations:

- **TPV-1**: rectangle placed to the east of center (`x` in [265, 320], `y` in [275, 325]).  
  The obstacle spans the east direction, blocking theta values whose east/west or perpendicular legs pass through it (including theta = 0 deg and theta = 90 deg).

- **TPV-2**: triangle placed above center (`x` near 500, `y` in [335, 390]).  
  The obstacle spans the north direction, blocking theta values whose north leg or east perpendicular passes through it (including theta = 0 deg and theta = 90 deg).

**Expected**: both cells visited; optimizer selects theta values that avoid all blocked orientations (thetas around 30-60 deg or 120-150 deg).

In [ ]:
# ─── Scenario 2 Setup ────────────────────────────────────────────────────────

# TPV-1: obstacle = CCW rectangle to the right of center.
# x in [265, 320], y in [275, 325].  The east leg (theta=0) reaches x~330 for
# L=130 and passes straight through this rectangle.
obs_s2_c1 = [
    [265.0, 275.0],
    [320.0, 275.0],
    [320.0, 325.0],
    [265.0, 325.0],
]

# TPV-2: obstacle = CCW triangle above center.
# The north leg (theta=90) reaches y~400 for L=120 and passes through this
# triangle.  The obstacle also blocks any cross whose perpendicular north leg
# passes through (e.g. theta=0 has a north perpendicular at x=500).
obs_s2_c2 = [
    [470.0, 335.0],
    [530.0, 335.0],
    [500.0, 390.0],
]

cells_s2 = [
    CircularSupercell(
        center=np.array([200.0, 300.0]),
        radius=50.0,
        L_max=130.0,
        weight=2.0,
        name='TPV-1',
        sensitivity_points=[],
        obstacle=obs_s2_c1
    ),
    CircularSupercell(
        center=np.array([500.0, 280.0]),
        radius=45.0,
        L_max=120.0,
        weight=1.5,
        name='TPV-2',
        sensitivity_points=[],
        obstacle=obs_s2_c2
    ),
]

# Budget set above the minimum feasible distance to visit both cells with
# valid (non-blocked) theta values (~1369 km)
BUDGET_S2 = 1600.0

print('Scenario 2 cells defined.')
for c in cells_s2:
    print(f'  {c.name}: center={c.center}, obstacle CCW vertices:')
    for v in c.obstacle:
        print(f'    {v}')

# Sanity checks
print()
print('Sanity checks (should all be True):')
print(f'  TPV-1 blocked at theta=0 deg, L=130  : '
      f'{cells_s2[0].cross_blocked(0.0, 130.0)}')
print(f'  TPV-2 blocked at theta=90 deg, L=120 : '
      f'{cells_s2[1].cross_blocked(np.pi / 2, 120.0)}')
print()
print('Available (unblocked) thetas per cell:')
for ci, cell in enumerate(cells_s2):
    avail = [round(np.degrees(th), 1)
             for th in THETA_GRID
             if not cell.cross_blocked(th, cell.L_max)]
    print(f'  {cell.name}: {avail}')

In [ ]:
# ─── Scenario 2 Optimize ─────────────────────────────────────────────────────

print('Running optimizer for Scenario 2...')
sol_s2 = optimize(cells_s2, BASE, BUDGET_S2, THETA_GRID, N_L_LEVELS=6)
print(f'  Best score  : {sol_s2["score"]:.2f}')
print(f'  Best dist   : {sol_s2["dist"]:.2f} km')
print(f'  Visit order : {sol_s2["order"]}')
print(f'  Thetas (deg): {[round(np.degrees(t), 1) for t in sol_s2["thetas"]]}')
print(f'  Ls          : {[round(L, 2) for L in sol_s2["Ls"]]}')

In [ ]:
# ─── Scenario 2 Plot ─────────────────────────────────────────────────────────

fig2, ax2 = plt.subplots(figsize=(9, 7))
draw_scenario(
    ax2, cells_s2, BASE, solution=sol_s2,
    title='Scenario 2: With Obstacles, No Sensitivity Points'
)
plt.tight_layout()
plt.savefig(
    '/home/yuanpei_cao/aerial-met-survey-optimizer/figures/unit_test_s2.png',
    dpi=120, bbox_inches='tight'
)
plt.show()
print('Figure saved.')

In [ ]:
# ─── Scenario 2 Summary Table ─────────────────────────────────────────────────

print('=' * 62)
print('SCENARIO 2 SUMMARY')
print('=' * 62)
print(f'{"Cell":<8} {"Visited":<9} {"Theta (deg)":<13} '
      f'{"L (km)":<10} {"Weight":<8} {"Blocked?":<10}')
print('-' * 62)
for i, cell in enumerate(cells_s2):
    visited   = 'YES' if i in sol_s2['order'] else 'NO'
    theta_deg = round(np.degrees(sol_s2['thetas'][i]), 1)
    L_val     = round(sol_s2['Ls'][i], 2)
    blocked   = cell.cross_blocked(sol_s2['thetas'][i], sol_s2['Ls'][i])
    print(f'{cell.name:<8} {visited:<9} {theta_deg:<13} '
          f'{L_val:<10} {cell.weight:<8} {str(blocked):<10}')
print('-' * 62)
print(f'{"TOTAL":<8} {"":<9} {"":<13} {"":<10} {sol_s2["score"]:<8.2f}')
print(f'Total route distance : {sol_s2["dist"]:.2f} km')
print(f'Budget               : {BUDGET_S2:.0f} km')
print(f'Budget utilization   : {sol_s2["dist"] / BUDGET_S2 * 100:.1f}%')
print(f'Note: Selected thetas must NOT be blocked (Blocked? = False).')
print('=' * 62)

---
## Scenario 3: With Obstacles AND Sensitivity Points

Same obstacles as Scenario 2, plus mandatory sensitivity points:

- **TPV-1**: two sensitivity points placed at the south-west (200 deg) and south-east (310 deg) of the circle boundary.  These are outside the obstacle area and force a post-scan detour around the south of the cell.
- **TPV-2**: one sensitivity point placed to the west (180 deg) of the circle boundary.

The full route must visit all sensitivity points after each cross scan using obstacle-avoiding segments.

**Expected**: both cells visited (score = 3.5); route distance is larger than Scenario 2 due to sensitivity-point detours; sensitivity points visibly appear as stars on the plot with the green route passing through them.

In [ ]:
# ─── Scenario 3 Setup ────────────────────────────────────────────────────────

c1_center = np.array([200.0, 300.0])
c1_radius = 50.0
c2_center = np.array([500.0, 280.0])
c2_radius = 45.0

# TPV-1 sensitivity points: south-west (200 deg) and south-east (310 deg)
# of the circle boundary.  Placed at radius+8 to sit just outside the circle,
# away from the east-side obstacle.
sp1_c1 = c1_center + (c1_radius + 8) * np.array(
    [cos(radians(200)), sin(radians(200))]
)   # south-west
sp2_c1 = c1_center + (c1_radius + 8) * np.array(
    [cos(radians(310)), sin(radians(310))]
)   # south-east

# TPV-2 sensitivity point: west (180 deg) of the circle boundary
sp1_c2 = c2_center + (c2_radius + 8) * np.array(
    [cos(radians(180)), sin(radians(180))]
)   # west

# Same obstacles as Scenario 2
obs_s3_c1 = [
    [265.0, 275.0],
    [320.0, 275.0],
    [320.0, 325.0],
    [265.0, 325.0],
]
obs_s3_c2 = [
    [470.0, 335.0],
    [530.0, 335.0],
    [500.0, 390.0],
]

cells_s3 = [
    CircularSupercell(
        center=c1_center,
        radius=c1_radius,
        L_max=130.0,
        weight=2.0,
        name='TPV-1',
        sensitivity_points=[sp1_c1, sp2_c1],
        obstacle=obs_s3_c1
    ),
    CircularSupercell(
        center=c2_center,
        radius=c2_radius,
        L_max=120.0,
        weight=1.5,
        name='TPV-2',
        sensitivity_points=[sp1_c2],
        obstacle=obs_s3_c2
    ),
]

# Budget set above the minimum feasible distance with sensitivity routing (~1518 km)
BUDGET_S3 = 1600.0

print('Scenario 3 cells defined.')
for c in cells_s3:
    print(f'  {c.name}: center={c.center}, '
          f'{len(c.sensitivity_points)} sensitivity point(s)')
    for j, sp in enumerate(c.sensitivity_points):
        print(f'    S{j + 1} = {sp.round(2)}')

In [ ]:
# ─── Scenario 3 Optimize ─────────────────────────────────────────────────────

print('Running optimizer for Scenario 3...')
sol_s3 = optimize(cells_s3, BASE, BUDGET_S3, THETA_GRID, N_L_LEVELS=6)
print(f'  Best score  : {sol_s3["score"]:.2f}')
print(f'  Best dist   : {sol_s3["dist"]:.2f} km')
print(f'  Visit order : {sol_s3["order"]}')
print(f'  Thetas (deg): {[round(np.degrees(t), 1) for t in sol_s3["thetas"]]}')
print(f'  Ls          : {[round(L, 2) for L in sol_s3["Ls"]]}')

In [ ]:
# ─── Scenario 3 Plot ─────────────────────────────────────────────────────────

fig3, ax3 = plt.subplots(figsize=(10, 8))
draw_scenario(
    ax3, cells_s3, BASE, solution=sol_s3,
    title='Scenario 3: With Obstacles AND Sensitivity Points'
)
plt.tight_layout()
plt.savefig(
    '/home/yuanpei_cao/aerial-met-survey-optimizer/figures/unit_test_s3.png',
    dpi=120, bbox_inches='tight'
)
plt.show()
print('Figure saved.')

In [ ]:
# ─── Scenario 3 Summary Table ─────────────────────────────────────────────────

print('=' * 65)
print('SCENARIO 3 SUMMARY')
print('=' * 65)
print(f'{"Cell":<8} {"Visited":<9} {"Theta (deg)":<13} '
      f'{"L (km)":<10} {"Weight":<8} {"Sens Pts":<10}')
print('-' * 65)
for i, cell in enumerate(cells_s3):
    visited   = 'YES' if i in sol_s3['order'] else 'NO'
    theta_deg = round(np.degrees(sol_s3['thetas'][i]), 1)
    L_val     = round(sol_s3['Ls'][i], 2)
    n_sp      = len(cell.sensitivity_points)
    print(f'{cell.name:<8} {visited:<9} {theta_deg:<13} '
          f'{L_val:<10} {cell.weight:<8} {n_sp:<10}')
print('-' * 65)
print(f'{"TOTAL":<8} {"":<9} {"":<13} {"":<10} {sol_s3["score"]:<8.2f}')
print(f'Total route distance : {sol_s3["dist"]:.2f} km')
print(f'Budget               : {BUDGET_S3:.0f} km')
print(f'Budget utilization   : {sol_s3["dist"] / BUDGET_S3 * 100:.1f}%')
print(f'Note: Post-scan sensitivity routing adds distance vs Scenario 2.')
print('=' * 65)

---
## Final Comparison

Side-by-side summary of all three scenarios and verification of expected outcomes.

In [ ]:
# ─── Final Comparison ────────────────────────────────────────────────────────

print('=' * 72)
print('ALL SCENARIOS COMPARISON')
print('=' * 72)
print(f'{"Scenario":<32} {"Score":<8} {"Dist (km)":<12} '
      f'{"Budget":<10} {"Util%":<8}')
print('-' * 72)

scenario_rows = [
    ('S1: No obs, no sens pts',     sol_s1, BUDGET_S1),
    ('S2: Obstacles, no sens pts',  sol_s2, BUDGET_S2),
    ('S3: Obstacles + sens pts',    sol_s3, BUDGET_S3),
]
for name, sol, budget in scenario_rows:
    util = sol['dist'] / budget * 100
    print(f'{name:<32} {sol["score"]:<8.2f} {sol["dist"]:<12.2f} '
          f'{budget:<10.0f} {util:<8.1f}')

print('=' * 72)
print()

# Verification assertions
assert sol_s1['score'] == 3.5, 'S1: expected both cells visited'
assert sol_s2['score'] == 3.5, 'S2: expected both cells visited'
assert sol_s3['score'] == 3.5, 'S3: expected both cells visited'
assert sol_s3['dist'] > sol_s2['dist'], \
    'S3 distance must exceed S2 (sensitivity point overhead)'
for i, cell in enumerate(cells_s2):
    assert not cell.cross_blocked(sol_s2['thetas'][i], sol_s2['Ls'][i]), \
        f'S2: {cell.name} chosen theta must not be blocked'
for i, cell in enumerate(cells_s3):
    assert not cell.cross_blocked(sol_s3['thetas'][i], sol_s3['Ls'][i]), \
        f'S3: {cell.name} chosen theta must not be blocked'

print('All verification assertions passed.')
print()
print('Summary of expected outcomes:')
print('  S1: Both cells visited (score=3.5). No obstacle constraints.')
print('  S2: Both cells visited. Thetas avoid blocked orientations '
      '(not near 0 or 90 deg).')
print('  S3: Both cells visited. Route detours to hit all sensitivity points.')
print(f'      Extra distance vs S2: {sol_s3["dist"] - sol_s2["dist"]:.1f} km')